# Multiple Regression Analysis
### Data Analysis for Business | Chapter 9

---

**Business question:** A retail chain wants to know which factors drive quarterly sales across its 200 stores, so it can allocate marketing budgets more effectively.

**Dataset:** `retail_marketing_regression.csv`  
**Variables:**
| Variable | Description |
|---|---|
| `quarterly_sales` | Quarterly sales revenue (£000s) — our **outcome** |
| `tv_spend` | TV advertising spend this quarter (£000s) |
| `online_spend` | Online/digital ad spend this quarter (£000s) |
| `store_size_sqft` | Store floor area (square feet) |
| `promo_weeks` | Number of promotional weeks in the quarter |

---
## Step 0 — Install and import libraries

In [ ]:
# Run this cell first to install any missing libraries
!pip install pandas numpy matplotlib seaborn scikit-learn scipy --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from scipy import stats

# Set plot style
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

print('Libraries loaded successfully!')

---
## Step 1 — Load and explore the data

In [ ]:
# Load the dataset
url = 'https://raw.githubusercontent.com/timesend/Datascience_work/main/retail_marketing_regression.csv'
df = pd.read_csv(url)

print(f'Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(10)

In [ ]:
# Descriptive statistics — get a feel for the data before modelling
df.describe().round(2)

In [ ]:
# Check for missing values
print('Missing values per column:')
print(df.isnull().sum())

---
## Step 2 — Visualise the relationships

Before building a model, it is good practice to look at how each predictor relates to the outcome.

In [ ]:
# Scatter plots of each predictor vs quarterly sales
predictors = ['tv_spend', 'online_spend', 'store_size_sqft', 'promo_weeks']
labels = ['TV Spend (£000s)', 'Online Spend (£000s)', 'Store Size (sq ft)', 'Promotional Weeks']
colors = ['#0D9488', '#1B2A4A', '#0D9488', '#1B2A4A']

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for i, (pred, label, color) in enumerate(zip(predictors, labels, colors)):
    axes[i].scatter(df[pred], df['quarterly_sales'], alpha=0.5, color=color, edgecolors='white', linewidth=0.3)
    # Add a simple trend line
    m, b = np.polyfit(df[pred], df['quarterly_sales'], 1)
    x_line = np.linspace(df[pred].min(), df[pred].max(), 100)
    axes[i].plot(x_line, m * x_line + b, color='#F59E0B', linewidth=2, label='Trend')
    axes[i].set_xlabel(label, fontsize=11)
    axes[i].set_ylabel('Quarterly Sales (£000s)', fontsize=11)
    axes[i].set_title(f'{label} vs Sales', fontsize=12, fontweight='bold')

plt.suptitle('Predictors vs Quarterly Sales', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

> **📝 Observation:** Do the scatter plots show a positive relationship between each predictor and sales? Does the slope look steep or shallow? Make a note before continuing.

In [ ]:
# Correlation matrix — useful before regression to spot highly correlated predictors
corr_vars = ['quarterly_sales', 'tv_spend', 'online_spend', 'store_size_sqft', 'promo_weeks']
corr = df[corr_vars].corr().round(2)

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Upper triangle only
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlGnBu',
            mask=mask, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCorrelation with quarterly_sales:')
print(corr['quarterly_sales'].drop('quarterly_sales').sort_values(ascending=False))

> **📝 Question:** Which predictor has the highest correlation with `quarterly_sales`? Are any two predictors highly correlated with *each other*? (High correlations between predictors — multicollinearity — can make coefficients harder to interpret.)

---
## Step 3 — State our hypotheses

Before running the model, we commit to what we expect to find.

| Hypothesis | Prediction | Rationale |
|---|---|---|
| **H1** — TV Spend | Positive association with sales | TV ads raise brand awareness and drive foot traffic |
| **H2** — Online Spend | Positive association with sales | Digital ads target customers actively searching to buy |
| **H3** — Store Size | Positive association with sales | More floor space means more product range |
| **H4** — Promo Weeks | Positive association with sales | Promotions incentivise purchases and increase basket size |

We will check each of these against the model output.

---
## Step 4 — Build the regression model

In [ ]:
# Define outcome variable (Y) and predictors (X)
X = df[['tv_spend', 'online_spend', 'store_size_sqft', 'promo_weeks']]
y = df['quarterly_sales']

print('Outcome (Y):   quarterly_sales')
print('Predictors (X):', list(X.columns))
print(f'Sample size:   {len(y)} stores')

In [ ]:
# Fit the multiple regression model
model = LinearRegression()
model.fit(X, y)

print('Model fitted!')
print(f'Intercept (constant): {model.intercept_:.2f}')
print('\nRaw coefficients:')
for name, coef in zip(X.columns, model.coef_):
    print(f'  {name:<22} {coef:.4f}')

---
## Step 5 — Generate the full regression summary

We need p-values and confidence intervals to complete our hypothesis tests. The code below computes these manually.

In [ ]:
def regression_summary(model, X, y):
    """Compute a full OLS regression summary table with p-values and confidence intervals."""
    X_arr = X.values
    y_arr = y.values
    n, p = X_arr.shape

    y_pred = model.predict(X)
    residuals = y_arr - y_pred

    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((y_arr - y_arr.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)

    X_b = np.column_stack([np.ones(n), X_arr])
    mse = ss_res / (n - p - 1)
    var_b = mse * np.linalg.inv(X_b.T @ X_b).diagonal()
    se_b = np.sqrt(var_b)

    coefs = np.concatenate([[model.intercept_], model.coef_])
    t_vals = coefs / se_b
    p_vals = [2 * (1 - stats.t.cdf(abs(t), df=n - p - 1)) for t in t_vals]
    ci_lo = coefs - 1.96 * se_b
    ci_hi = coefs + 1.96 * se_b

    ss_reg = ss_tot - ss_res
    f_stat = (ss_reg / p) / (ss_res / (n - p - 1))
    f_pval = 1 - stats.f.cdf(f_stat, p, n - p - 1)

    names = ['const'] + list(X.columns)

    # Print header
    print('=' * 68)
    print('                    OLS Regression Results')
    print('=' * 68)
    print(f'  R-squared:      {r2:.3f}       F-statistic:    {f_stat:.2f}')
    print(f'  Adj. R-squared: {adj_r2:.3f}       Prob(F-stat):   {f_pval:.2e}')
    print(f'  No. Obs:        {n}         Dep. Variable:  quarterly_sales')
    print('-' * 68)
    print(f'  {"Variable":<22} {"Coef":>8} {"p>|t|":>8}  {"[0.025":>9}  {"0.975]":>9}')
    print('-' * 68)
    for nm, c, p_v, cl, ch in zip(names, coefs, p_vals, ci_lo, ci_hi):
        sig = '***' if p_v < 0.001 else '**' if p_v < 0.01 else '*' if p_v < 0.05 else ''
        print(f'  {nm:<22} {c:>8.4f} {p_v:>8.4f}  {cl:>9.4f}  {ch:>9.4f}  {sig}')
    print('=' * 68)
    print('  Significance: *** p<0.001  ** p<0.01  * p<0.05')
    print('=' * 68)

    return {
        'r2': r2, 'adj_r2': adj_r2, 'f_stat': f_stat, 'f_pval': f_pval,
        'coefs': dict(zip(names, coefs)),
        'p_vals': dict(zip(names, p_vals)),
        'ci_lo': dict(zip(names, ci_lo)),
        'ci_hi': dict(zip(names, ci_hi)),
    }

results = regression_summary(model, X, y)

---
## Step 6 — Interpret the output

Use the table above to answer these questions:

### 6a — R-squared
> What percentage of the variation in quarterly sales across stores does our model explain?

In [ ]:
r2 = results['r2']
print(f'R-squared = {r2:.4f}')
print(f'Our model explains {r2*100:.1f}% of the variation in quarterly sales.')
print(f'The remaining {(1-r2)*100:.1f}% is driven by factors not in our model.')

### 6b — Coefficient interpretation
> For each predictor, what is the estimated change in sales for a one-unit increase?

In [ ]:
# Interpret each coefficient in plain English
interpretations = {
    'tv_spend': ('£1,000', 'quarterly sales'),
    'online_spend': ('£1,000', 'quarterly sales'),
    'store_size_sqft': ('1 sq ft', 'quarterly sales'),
    'promo_weeks': ('1 promotional week', 'quarterly sales'),
}

coefs = results['coefs']
print('Coefficient Interpretations')
print('=' * 70)
for var, (unit, outcome) in interpretations.items():
    coef = coefs[var]
    direction = 'increases' if coef > 0 else 'decreases'
    print(f'\n{var}:')
    print(f'  For every extra {unit}, {outcome} {direction} by £{abs(coef):.2f}k')
    print(f'  — holding all other variables constant.')

### 6c — P-value interpretation
> Which predictors are statistically significant at p < 0.05?

In [ ]:
p_vals = results['p_vals']
print('Significance Tests')
print('=' * 55)
print(f'{"Variable":<25} {"p-value":>10}  {"Significant?":>15}')
print('-' * 55)
for var in ['tv_spend', 'online_spend', 'store_size_sqft', 'promo_weeks']:
    p = p_vals[var]
    sig = 'YES (p < 0.001)' if p < 0.001 else 'YES (p < 0.05)' if p < 0.05 else 'NO'
    print(f'{var:<25} {p:>10.4f}  {sig:>15}')

---
## Step 7 — Hypothesis test verdicts

Now we return to our four hypotheses and check whether the data supports or rejects each one.

In [ ]:
hypotheses = [
    {
        'id': 'H1', 'var': 'tv_spend', 'label': 'TV Advertising',
        'prediction': 'positive coefficient, significant p-value'
    },
    {
        'id': 'H2', 'var': 'online_spend', 'label': 'Online Advertising',
        'prediction': 'positive coefficient, significant p-value'
    },
    {
        'id': 'H3', 'var': 'store_size_sqft', 'label': 'Store Size',
        'prediction': 'positive coefficient, significant p-value'
    },
    {
        'id': 'H4', 'var': 'promo_weeks', 'label': 'Promotional Weeks',
        'prediction': 'positive coefficient, significant p-value'
    },
]

print('HYPOTHESIS VERDICTS')
print('=' * 75)
for h in hypotheses:
    coef = results['coefs'][h['var']]
    p = results['p_vals'][h['var']]
    positive = coef > 0
    significant = p < 0.05
    supported = positive and significant
    verdict = '✓ SUPPORTED' if supported else '✗ NOT SUPPORTED'

    print(f"\n{h['id']} — {h['label']}")
    print(f"  Prediction: {h['prediction']}")
    print(f"  Result:     coef = {coef:.4f}  |  p = {p:.4f}")
    print(f"  Verdict:    {verdict}")

print('\n' + '=' * 75)

---
## Step 8 — Visualise the coefficient estimates

A coefficient plot shows each estimate and its 95% confidence interval — useful for comparing the relative strength of predictors.

In [ ]:
# Coefficient plot with confidence intervals
pred_vars = ['tv_spend', 'online_spend', 'store_size_sqft', 'promo_weeks']
pred_labels = ['TV Spend\n(£000s)', 'Online Spend\n(£000s)', 'Store Size\n(sq ft)', 'Promo\nWeeks']

coef_vals = [results['coefs'][v] for v in pred_vars]
ci_lo = [results['ci_lo'][v] for v in pred_vars]
ci_hi = [results['ci_hi'][v] for v in pred_vars]
errors_lo = [c - lo for c, lo in zip(coef_vals, ci_lo)]
errors_hi = [hi - c for c, hi in zip(coef_vals, ci_hi)]

colors = ['#0D9488' if c > 0 else '#EF4444' for c in coef_vals]

fig, ax = plt.subplots(figsize=(9, 5))
y_pos = range(len(pred_vars))

for i, (label, coef, lo, hi, color) in enumerate(zip(pred_labels, coef_vals, ci_lo, ci_hi, colors)):
    ax.barh(i, coef, color=color, alpha=0.8, height=0.5)
    ax.plot([lo, hi], [i, i], color='#1B2A4A', linewidth=2.5)
    ax.plot([lo, lo], [i - 0.1, i + 0.1], color='#1B2A4A', linewidth=2)
    ax.plot([hi, hi], [i - 0.1, i + 0.1], color='#1B2A4A', linewidth=2)
    ax.text(max(hi, coef) + 0.05, i, f' {coef:.3f}', va='center', fontsize=10, color='#1B2A4A', fontweight='bold')

ax.axvline(0, color='#475569', linewidth=1.2, linestyle='--')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(pred_labels, fontsize=11)
ax.set_xlabel('Coefficient (change in quarterly sales, £000s, per 1-unit increase)', fontsize=11)
ax.set_title('Regression Coefficients with 95% Confidence Intervals', fontsize=13, fontweight='bold')
ax.set_xlim(-1, max(coef_vals) + 1.5)
plt.tight_layout()
plt.show()

print('Note: Bars show the coefficient estimate. Lines show the 95% confidence interval.')
print('If the confidence interval does not cross zero, the effect is statistically significant.')

---
## Step 9 — Check residuals (optional, for reference)

Residuals are the gaps between our model's predictions and the actual values. A well-specified model should have residuals that look like random noise — no clear patterns.

In [ ]:
y_pred = model.predict(X)
residuals = y - y_pred

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Residuals vs fitted values
ax1.scatter(y_pred, residuals, alpha=0.5, color='#0D9488', edgecolors='white', linewidth=0.3)
ax1.axhline(0, color='#F59E0B', linewidth=2, linestyle='--')
ax1.set_xlabel('Fitted Values (predicted sales)', fontsize=11)
ax1.set_ylabel('Residuals', fontsize=11)
ax1.set_title('Residuals vs Fitted Values', fontsize=12, fontweight='bold')

# Distribution of residuals
ax2.hist(residuals, bins=25, color='#1B2A4A', edgecolor='white', alpha=0.85)
ax2.set_xlabel('Residuals', fontsize=11)
ax2.set_ylabel('Frequency', fontsize=11)
ax2.set_title('Distribution of Residuals', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Mean of residuals: {residuals.mean():.4f} (should be close to 0)')
print(f'Std of residuals:  {residuals.std():.2f} (£000s)')

> **📝 What to look for:** In the left plot, residuals should be scattered randomly around zero — no obvious fan shape or curve. In the right plot, residuals should look roughly bell-shaped. If either looks very wrong, the model assumptions may be violated.

---
## Step 10 — Business summary

Write your own plain-English summary of the findings using the template below.

In [ ]:
# Auto-generate a draft business summary — edit as you see fit!
coefs = results['coefs']
r2 = results['r2']

summary = f"""
BUSINESS SUMMARY
================

We modelled quarterly sales across {len(df)} stores using four predictors:
TV spend, online spend, store size, and promotional weeks.

Model fit: our model explains {r2*100:.1f}% of the variation in sales across stores (R² = {r2:.3f}).
This is strong fit for real-world business data.

Key findings:
  - TV advertising:    +£{coefs['tv_spend']:.0f} in sales per additional £1,000 spent
  - Online advertising: +£{coefs['online_spend']:.0f} in sales per additional £1,000 spent  ← strongest per-£ impact
  - Store size:         +£{coefs['store_size_sqft']*1000:.0f} in sales per additional 1,000 sq ft
  - Promotional weeks:  +£{coefs['promo_weeks']:.0f} in sales per additional promotional week

All four effects are statistically significant (p < 0.001).

Implication: Online advertising generates more sales per £ invested than TV.
However, this is an association, not proof of causation — an experiment would
be needed to confirm this before reallocating budget.
"""

print(summary)

---
---

# Part 2 — Employee Performance
## A Second Regression Example

Now we apply exactly the same steps to a different business problem. Work through this yourself — the structure mirrors Part 1 exactly.

---

**Business question:** An HR team wants to understand what drives employee performance scores, so it can target its investment in people more effectively.

**Dataset:** `employee_performance_regression.csv`  
**Variables:**
| Variable | Description |
|---|---|
| `performance_score` | Annual performance rating (0–100) — our **outcome** |
| `experience_years` | Total years of relevant work experience |
| `training_days` | Days of formal training completed this year |
| `team_size` | Number of people in the employee's team |
| `commute_mins` | One-way commute time in minutes |

> **⚠️ Important:** One of the four predictors may not behave as expected. Keep your hypotheses open-minded and let the data speak.

---
## Part 2 — Step 1: Load and explore the data

In [ ]:
# Load the employee performance dataset
url2 = 'https://raw.githubusercontent.com/timesend/Datascience_work/main/employee_performance_regression.csv'
df2 = pd.read_csv(url2)

print(f'Dataset shape: {df2.shape[0]} rows × {df2.shape[1]} columns')
df2.head(10)

In [ ]:
df2.describe().round(2)

In [ ]:
print('Missing values:')
print(df2.isnull().sum())

> **📝 Observation:** What is the average performance score? What is the range (min to max)? Does the spread look reasonable for a 0–100 scale?

---
## Part 2 — Step 2: Visualise the relationships

In [ ]:
predictors2 = ['experience_years', 'training_days', 'team_size', 'commute_mins']
labels2 = ['Experience (years)', 'Training (days/year)', 'Team Size (people)', 'Commute (minutes)']
colors2 = ['#0D9488', '#1B2A4A', '#0D9488', '#1B2A4A']

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.flatten()

for i, (pred, label, color) in enumerate(zip(predictors2, labels2, colors2)):
    axes[i].scatter(df2[pred], df2['performance_score'], alpha=0.5, color=color,
                    edgecolors='white', linewidth=0.3)
    m, b = np.polyfit(df2[pred], df2['performance_score'], 1)
    x_line = np.linspace(df2[pred].min(), df2[pred].max(), 100)
    axes[i].plot(x_line, m * x_line + b, color='#F59E0B', linewidth=2)
    axes[i].set_xlabel(label, fontsize=11)
    axes[i].set_ylabel('Performance Score', fontsize=11)
    axes[i].set_title(f'{label} vs Performance', fontsize=12, fontweight='bold')

plt.suptitle('Predictors vs Employee Performance Score', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

> **📝 Question:** Look carefully at the commute scatter plot. What direction is the slope? Does that match your intuition? Which predictor looks most weakly related to performance?

In [ ]:
corr_vars2 = ['performance_score', 'experience_years', 'training_days', 'team_size', 'commute_mins']
corr2 = df2[corr_vars2].corr().round(2)

fig, ax = plt.subplots(figsize=(7, 5))
mask2 = np.triu(np.ones_like(corr2, dtype=bool))
sns.heatmap(corr2, annot=True, fmt='.2f', cmap='YlGnBu',
            mask=mask2, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Matrix — Employee Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nCorrelation with performance_score:')
print(corr2['performance_score'].drop('performance_score').sort_values(ascending=False))

> **📝 Question:** Which predictor has the weakest correlation with `performance_score`? Does this give you any early signal about what the regression might show?

---
## Part 2 — Step 3: State your hypotheses

Fill in the table below **before** running the model. Predict both the direction (positive / negative) and whether you expect the effect to be statistically significant.

| Hypothesis | Your prediction | Your rationale |
|---|---|---|
| **H1** — Experience | *(positive / negative / unsure?)* | *(why?)* |
| **H2** — Training days | *(positive / negative / unsure?)* | *(why?)* |
| **H3** — Team size | *(positive / negative / unsure?)* | *(why?)* |
| **H4** — Commute | *(positive / negative / unsure?)* | *(why?)* |

> **Tip:** Unlike Part 1, the direction is not obvious for all predictors. Think about whether a longer commute might affect energy and focus, or whether team size has a clear relationship with individual performance.

---
## Part 2 — Step 4: Build the regression model

In [ ]:
X2 = df2[['experience_years', 'training_days', 'team_size', 'commute_mins']]
y2 = df2['performance_score']

model2 = LinearRegression()
model2.fit(X2, y2)

print('Model fitted!')
print(f'Intercept: {model2.intercept_:.2f}')
print('\nRaw coefficients:')
for name, coef in zip(X2.columns, model2.coef_):
    print(f'  {name:<22} {coef:.4f}')

---
## Part 2 — Step 5: Generate the full regression summary

In [ ]:
results2 = regression_summary(model2, X2, y2)

---
## Part 2 — Step 6: Interpret the output

### 6a — R-squared

In [ ]:
r2_2 = results2['r2']
print(f'R-squared = {r2_2:.4f}')
print(f'The model explains {r2_2*100:.1f}% of the variation in performance scores.')
print()
print('Compare with Part 1 (retail sales model):')
print(f'  Part 1 R² = {results["r2"]:.4f}  ({results["r2"]*100:.1f}% explained)')
print(f'  Part 2 R² = {r2_2:.4f}  ({r2_2*100:.1f}% explained)')
print()
print('Which model fits the data better? What might explain the difference?')

### 6b — Coefficient interpretation

In [ ]:
coefs2 = results2['coefs']

interpretations2 = {
    'experience_years': ('1 extra year of experience', 'performance score'),
    'training_days':    ('1 extra training day',       'performance score'),
    'team_size':        ('1 extra team member',        'performance score'),
    'commute_mins':     ('1 extra minute of commute',  'performance score'),
}

print('Coefficient Interpretations')
print('=' * 70)
for var, (unit, outcome) in interpretations2.items():
    coef = coefs2[var]
    direction = 'increases' if coef > 0 else 'decreases'
    print(f'\n{var}:')
    print(f'  For every {unit}, {outcome} {direction} by {abs(coef):.4f} points')
    print(f'  — holding all other variables constant.')

> **📝 Note on small coefficients:** The commute coefficient looks small (around −0.13 points per minute). But a 30-minute longer commute would correspond to ~4 fewer performance points. Think about whether that is meaningful in context.

---
## Part 2 — Step 7: Hypothesis verdicts

Run the cell below and compare the verdicts to your hypotheses in Step 3.

In [ ]:
hypotheses2 = [
    {'id': 'H1', 'var': 'experience_years', 'label': 'Work Experience',
     'prediction': 'positive coefficient, significant p-value'},
    {'id': 'H2', 'var': 'training_days',    'label': 'Training Days',
     'prediction': 'positive coefficient, significant p-value'},
    {'id': 'H3', 'var': 'team_size',        'label': 'Team Size',
     'prediction': 'to be filled in from your Step 3 table'},
    {'id': 'H4', 'var': 'commute_mins',     'label': 'Commute Time',
     'prediction': 'to be filled in from your Step 3 table'},
]

print('HYPOTHESIS VERDICTS — Employee Performance Model')
print('=' * 75)
for h in hypotheses2:
    coef = results2['coefs'][h['var']]
    p    = results2['p_vals'][h['var']]
    positive    = coef > 0
    significant = p < 0.05
    supported   = significant  # Note: we don't fix a direction for H3/H4 here
    verdict = '✓ SIGNIFICANT' if significant else '✗ NOT SIGNIFICANT'

    print(f"\n{h['id']} — {h['label']}")
    print(f"  Result:     coef = {coef:>8.4f}  |  p = {p:.4f}")
    print(f"  Direction:  {'positive ↑' if positive else 'negative ↓'}")
    print(f"  Verdict:    {verdict}")

print('\n' + '=' * 75)
print('\n❓ Which result surprised you most? Did any hypothesis fail to be supported?')

---
## Part 2 — Step 8: Visualise the coefficients

In [ ]:
pred_vars2   = ['experience_years', 'training_days', 'team_size', 'commute_mins']
pred_labels2 = ['Experience\n(years)', 'Training\n(days)', 'Team Size\n(people)', 'Commute\n(minutes)']

coef_vals2 = [results2['coefs'][v] for v in pred_vars2]
ci_lo2     = [results2['ci_lo'][v] for v in pred_vars2]
ci_hi2     = [results2['ci_hi'][v] for v in pred_vars2]
colors_c2  = ['#0D9488' if c > 0 else '#EF4444' for c in coef_vals2]

fig, ax = plt.subplots(figsize=(9, 5))
y_pos2 = range(len(pred_vars2))

for i, (label, coef, lo, hi, color) in enumerate(
        zip(pred_labels2, coef_vals2, ci_lo2, ci_hi2, colors_c2)):
    ax.barh(i, coef, color=color, alpha=0.8, height=0.5)
    ax.plot([lo, hi], [i, i], color='#1B2A4A', linewidth=2.5)
    ax.plot([lo, lo], [i-0.1, i+0.1], color='#1B2A4A', linewidth=2)
    ax.plot([hi, hi], [i-0.1, i+0.1], color='#1B2A4A', linewidth=2)
    ax.text(max(abs(hi), abs(coef)) * (1 if coef >= 0 else -1) + 0.05,
            i, f' {coef:.3f}', va='center', fontsize=10, color='#1B2A4A', fontweight='bold')

ax.axvline(0, color='#475569', linewidth=1.5, linestyle='--')
ax.set_yticks(list(y_pos2))
ax.set_yticklabels(pred_labels2, fontsize=11)
ax.set_xlabel('Coefficient (change in performance score per 1-unit increase)', fontsize=11)
ax.set_title('Regression Coefficients — Employee Performance Model\nwith 95% Confidence Intervals',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Green bars = positive effect.  Red bars = negative effect.')
print('If the confidence interval crosses zero, the effect is NOT statistically significant.')

> **📝 Key observation:** One of the confidence intervals crosses zero — that predictor's effect is not statistically distinguishable from zero. Which one is it? What does that mean for the HR team?

---
## Part 2 — Step 9: Check residuals

In [ ]:
y_pred2   = model2.predict(X2)
residuals2 = y2 - y_pred2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(y_pred2, residuals2, alpha=0.5, color='#0D9488', edgecolors='white', linewidth=0.3)
ax1.axhline(0, color='#F59E0B', linewidth=2, linestyle='--')
ax1.set_xlabel('Fitted Values (predicted score)', fontsize=11)
ax1.set_ylabel('Residuals', fontsize=11)
ax1.set_title('Residuals vs Fitted — Employee Model', fontsize=12, fontweight='bold')

ax2.hist(residuals2, bins=25, color='#1B2A4A', edgecolor='white', alpha=0.85)
ax2.set_xlabel('Residuals', fontsize=11)
ax2.set_ylabel('Frequency', fontsize=11)
ax2.set_title('Distribution of Residuals', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Mean of residuals: {residuals2.mean():.4f}')
print(f'Std of residuals:  {residuals2.std():.2f} performance points')

> **📝 Notice:** You may see some compression at the top of the residuals plot — that's because the performance score is capped at 100. High-performing employees are 'bunched up' at the ceiling. How might this affect the model's reliability for very high scorers?

---
## Part 2 — Step 10: Business summary

Write your own interpretation in the cell below. The template gives you a starting point — edit it to reflect what you actually found.

In [ ]:
coefs2_d = results2['coefs']
r2_2     = results2['r2']
p_ts     = results2['p_vals']

team_sig = 'significant' if p_ts['team_size'] < 0.05 else 'NOT significant'

summary2 = f"""
BUSINESS SUMMARY — Employee Performance Model
=============================================

We modelled performance scores across {len(df2)} employees using four predictors:
experience, training days, team size, and commute time.

Model fit: R² = {r2_2:.3f} — the model explains {r2_2*100:.1f}% of the variation
in performance scores. (Compare: Part 1 retail model explained {results['r2']*100:.1f}%.)

Key findings:
  - Experience:   +{coefs2_d['experience_years']:.2f} points per additional year of experience
  - Training:     +{coefs2_d['training_days']:.2f} points per additional training day per year
  - Team size:    {coefs2_d['team_size']:+.4f} points per additional team member ({team_sig}, p={p_ts['team_size']:.3f})
  - Commute:      {coefs2_d['commute_mins']:.2f} points per additional minute of commute

Recommendation (draft — edit based on your analysis):
  Investing in training days and supporting employees to reduce commute times
  (e.g. flexible working) appear to be the most actionable levers for the HR team.
  Team size does not appear to be a meaningful driver of individual performance.

Caveat: This is observational data. We cannot conclude that shorter commutes
cause better performance without experimental evidence.
"""

print(summary2)

---
## Part 2 — 🎯 Exercise Questions

Answer these questions using your output from Steps 4–7 above.

**Q1.** What does the R² for this model tell you? Is it higher or lower than the retail model in Part 1? Why might employee performance be harder or easier to explain with four variables than store sales?

**Q2.** An employee completes 10 more training days this year than a colleague, but everything else is the same. By how much would the model predict their performance score to differ?

**Q3.** The commute coefficient is negative. Interpret this in plain English for an HR manager who has never studied statistics.

**Q4.** One predictor is not statistically significant (p > 0.05). Which one is it, and what should the HR team conclude? Should they remove it from the model?

**Q5.** Employee A has 5 years' experience, 10 training days, a team of 8, and a 20-minute commute. Use the regression equation to predict their performance score:

```
Score = 49.62 + 2.26×experience + 0.92×training + 0.01×team − 0.13×commute
```

**Q6.** A manager argues: *'We should only hire people who live close to the office, because commute time reduces performance.'* Critically evaluate this claim using what you have learned about regression and causation.

---
## Part 2 — 🏆 Extension Tasks

**E1 — Remove the insignificant predictor.** Drop `team_size` and refit the model. Does R² change much? Do the remaining coefficients shift? What does this tell you about the role of insignificant variables?

**E2 — Compare models side by side.** Print the R², F-statistic, and all four coefficients for both the retail model (Part 1) and the performance model (Part 2) in a single table. Which model fits better? Which has more practically useful coefficients?

**E3 — What is the ceiling effect?** Run `df2['performance_score'].value_counts().sort_index().tail(10)` and look at how many employees scored 100. What percentage hit the ceiling? Why is this a problem for OLS regression?

**E4 — Practical impact of training.** If the HR team could increase each employee's training days from the current average by 5 days, what would be the predicted average increase in performance score across all 250 employees?

---
## 🎯 Extension tasks

If you finish early, try one or more of these:

1. **What if we drop a predictor?** Rerun the model with only `tv_spend` and `online_spend`. Does the R² fall much? Do the coefficients change?

2. **What does a store predict?** Pick any store in the dataset. Use the regression equation to compute its predicted sales manually: `Sales = 74.02 + 3.64×tv + 5.06×online + 0.013×size + 4.06×promos`. How close is it to the actual value?

3. **Compare online vs TV ROI.** If a store has £10,000 to spend on advertising and can split it any way between TV and online, which split maximises predicted sales? (Hint: compare coefficients!)

4. **Scatter of predicted vs actual.** Plot `y_pred` on the x-axis and `y` on the y-axis. A perfect model would give a 45° line. How close is yours?